# 04 U-Net++ Training

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

## Dataset Preparation

In [ ]:
!pip install safetensors huggingface_hub

In [ ]:
import os
import copy
import torch
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp

from safetensors.torch import save_file
from torch.utils.data import DataLoader
from src.dataset.dataset import Dataset as LazyDS
from src.dataset.dataloader import DataLoader as DSLoader
from src.dataset.mben import MBENUNetPlusPlus
from src.models.unetpp import Train

In [ ]:
IMG_SIZE = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE  = 8 
LR          = 1e-4
EPOCHS      = 50
PATIENCE    = 10           # early stopping patience
CHECKPOINT  = 'best_unetpp.pth'
NUM_WORKERS = 2

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
loader = DSLoader(mask_folder       = "GTA - Masks",
                prnu_folder         = "Feature - PRNU",
                illumination_folder = "Feature - Illumination",
                frequency_folder    = "Feature - Frequency", 
                categories          = ['Synthetic', 'Authentic'], 
                templates           = ['template-a', 'template-albania', 'template-b', 
                                        'template-c', 'template-chile', 'template-deutschland', 
                                        'template-spain', 'template-usa'])

In [ ]:
train_samples = loader.load_images('Train', DATASET_ROOT)
val_samples   = loader.load_images('Validation', DATASET_ROOT)

train_ds = LazyDS(train_samples, IMG_SIZE, augment=True)
val_ds = LazyDS(val_samples, IMG_SIZE, augment=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                        num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## MBEN + U-Net++ Model

Architecture overview:
```
PRNU  (1,H,W) ──► Branch-1 CNN ──┐
Illu  (1,H,W) ──► Branch-2 CNN ──┼──► Element-wise SUM  ──┐
Freq  (1,H,W) ──► Branch-3 CNN ──┘                         ├──► Concat ──► Projection Conv ──► U-Net++
Fused (3,H,W) ──► Concat Stem ─────────────────────────────┘
```
- **MBEN branches**: three independent CNN encoders let each feature learn its own representation before being summed.
- **Concat stem**: a shallow CNN on the stacked 3-channel input preserves inter-feature spatial relationships.
- **Projection**: merges both paths into 3 channels for the EfficientNet-B4 backbone of U-Net++.

In [ ]:
MBEN_OUT_CHANNELS = 64

model = MBENUNetPlusPlus(mben_out_ch=MBEN_OUT_CHANNELS).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## Loss, Optimizer, Scheduler

In [ ]:
dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

## Training Loop

In [ ]:
best_val_loss      = float('inf')
best_model_wts     = copy.deepcopy(model.state_dict())
early_stop_counter = 0
train_losses, val_losses = [], []

train = Train(device, dice_loss)

In [ ]:
for epoch in range(1, EPOCHS + 1):
    train_loss = train.run_epoch(train_loader, model, optimizer, train=True)
    val_loss   = train.run_epoch(val_loader,   model, train=False)

    scheduler.step(val_loss)
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f'Epoch [{epoch:03d}/{EPOCHS}]  '
          f'Train Loss: {train_loss:.4f}  |  Val Loss: {val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss      = val_loss
        best_model_wts     = copy.deepcopy(model.state_dict())
        early_stop_counter = 0
        save_file({k: v.cpu() for k, v in best_model_wts.items()}, 'model.safetensors')
        print(f'  ✔ New best val loss: {best_val_loss:.4f} — checkpoint saved.')
    else:
        early_stop_counter += 1
        print(f'  No improvement ({early_stop_counter}/{PATIENCE})')
        if early_stop_counter >= PATIENCE:
            print('Early stopping triggered.')
            break

model.load_state_dict(best_model_wts)
print(f'\nTraining complete. Best Val Loss: {best_val_loss:.4f}')

## Loss Curves

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train Dice Loss')
plt.plot(val_losses,   label='Val Dice Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MBEN + U-Net++ Training & Validation Loss')
plt.legend()
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150)
plt.show()

## Quick Inference Check

In [ ]:
model.eval()
prnu, illu, freq, fused, masks = next(iter(val_loader))
prnu, illu, freq, fused = prnu.to(device), illu.to(device), freq.to(device), fused.to(device)

with torch.no_grad():
    preds = torch.sigmoid(model(prnu, illu, freq, fused)).cpu().numpy()

fig, axes = plt.subplots(4, 4, figsize=(14, 13))
for i in range(min(4, prnu.size(0))):
    axes[0, i].imshow(prnu[i, 0].cpu().numpy(),  cmap='gray'); axes[0, i].set_title('PRNU')
    axes[1, i].imshow(illu[i, 0].cpu().numpy(),  cmap='gray'); axes[1, i].set_title('Illumination')
    axes[2, i].imshow(masks[i, 0].numpy(),        cmap='gray'); axes[2, i].set_title('Ground Truth')
    axes[3, i].imshow(preds[i, 0],                cmap='gray'); axes[3, i].set_title('Prediction')

for ax in axes.flatten():
    ax.axis('off')
plt.tight_layout()
plt.savefig('inference_sample.png', dpi=150)
plt.show()

## Save Final Model 

### Save to Hugging Face Repository 

In [ ]:
from huggingface_hub import HfApi

api = HfApi()

# Create the repo first (only needed once)
username = 'hf-username'
api.create_repo(repo_id=f"{username}/mben-unetpp", exist_ok=True)

# Upload the weights
api.upload_file(
    path_or_fileobj='model.safetensors',
    path_in_repo='model.safetensors',
    repo_id="your-username/mben-unetpp",
)

In [ ]:
import json

config = {
    "model_type": "MBENUNetPlusPlus",
    "mben_out_ch": 64,
    "encoder": "efficientnet-b4",
    "encoder_weights": "imagenet",
    "in_channels": 3,
    "classes": 1,
}

with open("config.json", "w") as f:
    json.dump(config, f, indent=2)

api.upload_file(
    path_or_fileobj="config.json",
    path_in_repo="config.json",
    repo_id="your-username/mben-unetpp",
)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # opens a token input widget

### Save to Google Drive

In [ ]:
import shutil
gdrive_save_dir = '/content/drive/MyDrive/UNetPP_Checkpoints'
os.makedirs(gdrive_save_dir, exist_ok=True)

shutil.copy(CHECKPOINT,             os.path.join(gdrive_save_dir, CHECKPOINT))
shutil.copy('loss_curve.png',       os.path.join(gdrive_save_dir, 'loss_curve.png'))
shutil.copy('inference_sample.png', os.path.join(gdrive_save_dir, 'inference_sample.png'))

print(f'Model & plots saved to {gdrive_save_dir}')

## Load model from Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file

# Download weights
username = 'hf-username'
weights_path = hf_hub_download(repo_id=f"{username}/mben-unetpp", filename="model.safetensors")
state_dict = load_file(weights_path)

# Reconstruct model
model = MBENUNetPlusPlus(mben_out_ch=64)
model.load_state_dict(state_dict)